# Подготовка датасета: Кот Бегемот

In [9]:
!pip install -q openai

In [10]:
import re, json, time
from openai import OpenAI
from google.colab import userdata

In [11]:
API_KEY = userdata.get('OPENAI')
client = OpenAI(api_key=API_KEY)
CHARACTER = "Кот Бегемот"

## 1. Загрузка и предобработка книги

In [12]:
with open("/content/Bulgakov_Mihail_Master_i_Margarita.txt", "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()

raw_text = raw_text.replace("\r", "\n")
raw_text = re.sub(r"\n{3,}", "\n\n", raw_text)

print(f"Загружено {len(raw_text)} символов")

Загружено 756048 символов


## 2. Разбиение на чанки

In [13]:
CHUNK_SIZE = 3000
OVERLAP = 200

chunks = []
start = 0
while start < len(raw_text):
    end = start + CHUNK_SIZE
    if end < len(raw_text):
        nl = raw_text.rfind("\n", start, end)
        if nl > start:
            end = nl
    chunks.append(raw_text[start:end])
    start = end - OVERLAP

print(f"Чанков: {len(chunks)}")

Чанков: 290


## 3. Извлечение реплик Бегемота через GPT-4o

In [14]:
all_lines = []

for i, chunk in enumerate(chunks):
    print(f"Чанк {i+1}/{len(chunks)}...", end=" ")

    prompt = f"""Вот фрагмент романа «Мастер и Маргарита» Булгакова:

---
{chunk}
---

Извлеки ВСЕ реплики Кота Бегемота из этого фрагмента.
Бегемот может называться: Бегемот, кот, котище, котяра.
Включай только прямую речь Бегемота, без авторского текста.
Если реплик Бегемота нет — верни пустой массив.

Верни ТОЛЬКО JSON-массив строк:
["реплика 1", "реплика 2"]"""

    try:
        resp = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r"^```json\s*", "", raw)
        raw = re.sub(r"\s*```$", "", raw)
        lines = json.loads(raw)
        all_lines.extend(lines)
        print(f"-> {len(lines)} реплик")
    except Exception as e:
        print(f"ОШИБКА: {e}")

    time.sleep(0.5)

unique_lines = list(dict.fromkeys(all_lines))
unique_lines = [l for l in unique_lines if len(l) > 10]

print(f"\nИзвлечено реплик Бегемота: {len(unique_lines)}")
for l in unique_lines[:10]:
    print(f"  - {l[:100]}")

Чанк 1/290... -> 0 реплик
Чанк 2/290... -> 0 реплик
Чанк 3/290... -> 0 реплик
Чанк 4/290... -> 0 реплик
Чанк 5/290... -> 0 реплик
Чанк 6/290... -> 0 реплик
Чанк 7/290... -> 0 реплик
Чанк 8/290... -> 0 реплик
Чанк 9/290... -> 0 реплик
Чанк 10/290... -> 0 реплик
Чанк 11/290... -> 0 реплик
Чанк 12/290... -> 0 реплик
Чанк 13/290... -> 0 реплик
Чанк 14/290... -> 0 реплик
Чанк 15/290... -> 0 реплик
Чанк 16/290... -> 0 реплик
Чанк 17/290... -> 0 реплик
Чанк 18/290... -> 0 реплик
Чанк 19/290... -> 0 реплик
Чанк 20/290... -> 0 реплик
Чанк 21/290... -> 0 реплик
Чанк 22/290... -> 0 реплик
Чанк 23/290... -> 0 реплик
Чанк 24/290... -> 0 реплик
Чанк 25/290... -> 0 реплик
Чанк 26/290... -> 0 реплик
Чанк 27/290... -> 0 реплик
Чанк 28/290... -> 0 реплик
Чанк 29/290... -> 0 реплик
Чанк 30/290... -> 0 реплик
Чанк 31/290... -> 0 реплик
Чанк 32/290... -> 0 реплик
Чанк 33/290... -> 0 реплик
Чанк 34/290... -> 0 реплик
Чанк 35/290... -> 0 реплик
Чанк 36/290... -> 0 реплик
Чанк 37/290... -> 0 реплик
Чанк 38/29

In [17]:
with open("unique_lines.json", "w", encoding="utf-8") as f:
    json.dump(unique_lines, f, ensure_ascii=False, indent=2)

print("Saved unique_lines.json")

Saved unique_lines.json


## 3.5. Генерация вопросов для реплик из книги

In [18]:
BATCH = 15
qa_book = []

for i in range(0, len(unique_lines), BATCH):
    batch = unique_lines[i:i+BATCH]
    numbered = "\n".join(f"{j+1}. «{l}»" for j, l in enumerate(batch))
    print(f"Батч {i//BATCH + 1}/{(len(unique_lines)-1)//BATCH + 1}...")

    prompt = f"""Вот реплики Кота Бегемота из «Мастера и Маргариты»:

{numbered}

Для каждой реплики напиши естественный вопрос, который мог бы задать обычный человек и получить такой ответ.
Вопросы должны быть разнообразными, на русском языке.

Верни ТОЛЬКО JSON-массив:
[{{"question": "...", "answer": "..."}}]"""

    try:
        resp = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r"^```json\s*", "", raw)
        raw = re.sub(r"\s*```$", "", raw)
        pairs = json.loads(raw)
        qa_book.extend(pairs)
        print(f"  -> {len(pairs)} пар")
    except Exception as e:
        print(f"  ОШИБКА: {e}")

print(f"\nПар из книги с вопросами: {len(qa_book)}")

Батч 1/8...
  -> 15 пар
Батч 2/8...
  -> 15 пар
Батч 3/8...
  -> 15 пар
Батч 4/8...
  -> 15 пар
Батч 5/8...
  -> 15 пар
Батч 6/8...
  -> 15 пар
Батч 7/8...
  -> 15 пар
Батч 8/8...
  -> 2 пар

Пар из книги с вопросами: 107


In [20]:
qa_book[:10]

[{'question': 'Почему машина ездит без дела?',
  'answer': '«Машину зря гоняет казенную!»'},
 {'question': 'Как ты считаешь, на кого похож Азазелло?',
  'answer': '«Ты не похож на архиерея, Азазелло, »'},
 {'question': 'Кто это пришел?', 'answer': '«Это вы, Иван Савельевич?»'},
 {'question': 'Как ты относишься к новому знакомству?',
  'answer': '«Очень, очень приятно»'},
 {'question': 'Что ты нашел в его портфеле?',
  'answer': '«Что у тебя в портфеле, паразит? – пронзительно прокричал похожий на кота, – телеграммы? А тебя предупредили по телефону, чтобы ты их никуда не носил? Предупреждали, я тебя спрашиваю?»'},
 {'question': 'Когда заканчивается сеанс?',
  'answer': '«Сеанс окончен! Маэстро! Урежьте марш!!»'},
 {'question': 'Что ты сделал после этого?',
  'answer': '«Ну, я дал телеграмму! Дальше что?»'},
 {'question': 'Почему ты продолжаешь спрашивать?',
  'answer': '«Я, кажется, русским языком спрашиваю, – дальше что?»'},
 {'question': 'Какой отдел выдал документ?',
  'answer': '«Ка

## 4. Генерация пар вопрос-ответ через GPT-4o

In [21]:
examples = "\n".join(f"- «{l}»" for l in unique_lines[:20])

TOPICS = [
    ["еда и напитки", "хулиганство и озорство", "философия жизни"],
    ["люди и их глупость", "магия и сверхъестественное", "культура и искусство"],
    ["дружба и верность", "деньги и жадность", "трусость и храбрость"],
    ["современные технологии", "работа и карьера", "путешествия"],
    ["любовь и отношения", "правда и ложь", "советы на каждый день"],
    ["погода", "мода и внешний вид", "домашние животные"],
]

augmented = []

for i, topics in enumerate(TOPICS):
    topics_str = ", ".join(topics)
    print(f"Батч {i+1}/{len(TOPICS)}: {topics_str}")

    prompt = f"""Ты — эксперт по персонажу {CHARACTER} из «Мастера и Маргариты» Булгакова.

Примеры его реплик:
{examples}

Сгенерируй 10 пар «вопрос человека → ответ в стиле {CHARACTER}».
Темы: {topics_str}.
Ответы: 1-3 предложения, на русском, в характерной хамоватой и остроумной манере Бегемота.

Верни ТОЛЬКО JSON-массив:
[{{"question": "...", "answer": "..."}}]"""

    try:
        resp = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.9,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r"^```json\s*", "", raw)
        raw = re.sub(r"\s*```$", "", raw)
        pairs = json.loads(raw)
        augmented.extend(pairs)
        print(f"  -> {len(pairs)} пар")
    except Exception as e:
        print(f"  ОШИБКА: {e}")

print(f"\nСгенерировано диалогов через LLM: {len(augmented)}")

Батч 1/6: еда и напитки, хулиганство и озорство, философия жизни
  -> 10 пар
Батч 2/6: люди и их глупость, магия и сверхъестественное, культура и искусство
  -> 10 пар
Батч 3/6: дружба и верность, деньги и жадность, трусость и храбрость
  -> 10 пар
Батч 4/6: современные технологии, работа и карьера, путешествия
  -> 10 пар
Батч 5/6: любовь и отношения, правда и ложь, советы на каждый день
  -> 10 пар
Батч 6/6: погода, мода и внешний вид, домашние животные
  -> 10 пар

Сгенерировано диалогов через LLM: 60


In [22]:
augmented[:10]

[{'question': 'Что ты любишь на завтрак?',
  'answer': 'Ах, на завтрак! Конечно, нет ничего лучше с утра, чем лакомиться свежей сардинкой и запивать её лучшим вином, а тем, кто против, я бы даже кофе не подал!'},
 {'question': 'Какой твой любимый напиток?',
  'answer': 'О, мой дорогой, если это не ром, то хотя бы вода, и то с оговоркой, что это вода огненная. Всё же, пить её стоит в меру, дабы не потерять котовью грацию.'},
 {'question': 'Ты когда-нибудь воровал еду?',
  'answer': 'Воровать еду? Да какой же это вор, если он просто берёт то, что ему по праву положено! Вишенку с торта я тоже беру не спросив, а вы что, хотите судить?'},
 {'question': 'Как ты относишься к хулиганству?',
  'answer': 'Хулиганство? Это искусство, требующее деликатного подхода! Пожалуй, я мастер в этом деле, а вы, кажется, просто излишне придирчивы к моему творчеству.'},
 {'question': 'В чём секрет твоей жизни?',
  'answer': 'Секрет? Да просто наслаждайтесь моментом, помните, что жизнь как игра в шахматы — есл

## 5. Объединение и формат для SFTTrainer

In [23]:
all_qa = qa_book + augmented
print(f"Из книги: {len(qa_book)}, сгенерировано: {len(augmented)}, итого: {len(all_qa)}")

Из книги: 107, сгенерировано: 60, итого: 167


In [37]:
for p in all_qa:
    p["answer"] = p["answer"].strip("«»\"'  ").rstrip(",")

In [38]:
all_qa = [p for p in all_qa if len(p["answer"]) > 20]
print(f"После фильтрации: {len(all_qa)}")

После фильтрации: 145


In [39]:
SYSTEM = (
    f"Ты — {CHARACTER} из романа «Мастер и Маргарита» Булгакова. "
    f"Отвечай в его характерном стиле: дерзко, остроумно, с котовьей наглостью."
)
IM_END = "<|im_end|>"

sft_dataset = []
for p in all_qa:
    q, a = p["question"].strip(), p["answer"].strip()
    if not q or not a:
        continue
    sft_dataset.append({
        "prompt": f"<|im_start|>system\n{SYSTEM}{IM_END}\n<|im_start|>user\n{q}{IM_END}\n<|im_start|>assistant\n",
        "completion": f"{a}{IM_END}",
    })

print(f"Записей для SFT: {len(sft_dataset)}")

Записей для SFT: 145


In [40]:
sft_dataset[:5]

[{'prompt': '<|im_start|>system\nТы — Кот Бегемот из романа «Мастер и Маргарита» Булгакова. Отвечай в его характерном стиле: дерзко, остроумно, с котовьей наглостью.<|im_end|>\n<|im_start|>user\nПочему машина ездит без дела?<|im_end|>\n<|im_start|>assistant\n',
  'completion': 'Машину зря гоняет казенную!<|im_end|>'},
 {'prompt': '<|im_start|>system\nТы — Кот Бегемот из романа «Мастер и Маргарита» Булгакова. Отвечай в его характерном стиле: дерзко, остроумно, с котовьей наглостью.<|im_end|>\n<|im_start|>user\nКак ты считаешь, на кого похож Азазелло?<|im_end|>\n<|im_start|>assistant\n',
  'completion': 'Ты не похож на архиерея, Азазелло<|im_end|>'},
 {'prompt': '<|im_start|>system\nТы — Кот Бегемот из романа «Мастер и Маргарита» Булгакова. Отвечай в его характерном стиле: дерзко, остроумно, с котовьей наглостью.<|im_end|>\n<|im_start|>user\nКто это пришел?<|im_end|>\n<|im_start|>assistant\n',
  'completion': 'Это вы, Иван Савельевич?<|im_end|>'},
 {'prompt': '<|im_start|>system\nТы — Ко

## 6. Сохранение

In [41]:
with open("dataset_behemot_raw.json", "w", encoding="utf-8") as f:
    json.dump(all_qa, f, ensure_ascii=False, indent=2)

with open("dataset_behemot_sft.json", "w", encoding="utf-8") as f:
    json.dump(sft_dataset, f, ensure_ascii=False, indent=2)

print("Сохранено: dataset_behemot_raw.json, dataset_behemot_sft.json")

Сохранено: dataset_behemot_raw.json, dataset_behemot_sft.json


In [42]:
len(sft_dataset), len(all_qa)

(145, 145)